In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_table = "retailnova.bronze.customers"
silver_table = "retailnova.silver.customers"

In [0]:
df = spark.table(bronze_table)

In [0]:
# Get last processed timestamp for Silver
last_processed = spark.sql("""
    SELECT last_processed_at
    FROM retailnova.bronze.etl_control
    WHERE source_name = 'customers_silver'
""").first()[0]

print("Last Silver processed:", last_processed)

Last Silver processed: 1900-01-01 00:00:00


In [0]:
# Keep only new/changed customer records
new_customers = df.filter(
    col("updated_at") > last_processed
)

print("New records for Silver:", new_customers.count())

New records for Silver: 105000


In [0]:
# Remove records without customer_id
silver_df = new_customers.filter(
    col("customer_id").isNotNull()
)


In [0]:
# Standardize city
silver_df = silver_df.withColumn(
    "city",
    trim(col("city"))
)

In [0]:
# Standardize state
silver_df = silver_df.withColumn(
    "state",
    trim(col("state"))
)

In [0]:
# Standardize customer tier
silver_df = silver_df.withColumn(
    "customer_tier",
    trim(col("customer_tier"))
)

In [0]:
# First run creates the table
if not spark.catalog.tableExists(silver_table):

    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(silver_table)

else:

    silver_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

In [0]:
# Get the actual latest timestamp in Silver
new_watermark = spark.sql(f"""
    SELECT MAX(updated_at)
    FROM {silver_table}
""").first()[0]

print("New Silver watermark:", new_watermark)


New Silver watermark: 2026-09-10 23:59:10


In [0]:
# Update Silver watermark
spark.sql(f"""
    UPDATE retailnova.bronze.etl_control
    SET last_processed_at = TIMESTAMP('{new_watermark}')
    WHERE source_name = 'customers_silver'
""")

print("Silver customers updated successfully")

Silver customers updated successfully
